In [0]:
%sql
create catalog if not exists queries;


In [0]:
%sql
use catalog queries;

create table if not exists table1 (
  user_id int,
  activity_date date,
  value int
);

insert into table1
values 
  (1, '2022-01-01', 10),
  (1, '2022-01-02', 20),
  (1, '2022-01-03', 30),
  (2, '2022-01-01', 10),
  (2, '2022-01-02', 8),
  (2, '2022-01-03', 15),
  (3, '2022-01-01', 10),
  (3, '2022-01-02', 10),
  (3, '2022-01-03', 20),
  (4, '2022-01-01', 50);

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
select * from queries.default.table1

user_id,activity_date,value
1,2022-01-01,10
1,2022-01-02,20
1,2022-01-03,30
2,2022-01-01,10
2,2022-01-02,8
2,2022-01-03,15
3,2022-01-01,10
3,2022-01-02,10
3,2022-01-03,20
4,2022-01-01,50


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5069726988480520>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select *,\ncase when next_col>value then 1 else 0 end as flag\n from queries.default.table1\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseEx

In [0]:
%sql
select *,lead(value,1) over(partition by user_id order by activity_date) as next_col from queries.default.table1

user_id,activity_date,value,next_col
1,2022-01-01,10,20
1,2022-01-02,20,30
1,2022-01-03,30,null
2,2022-01-01,10,8
2,2022-01-02,8,15
2,2022-01-03,15,null
3,2022-01-01,10,10
3,2022-01-02,10,20
3,2022-01-03,20,null
4,2022-01-01,50,null


In [0]:
%sql
with t1 as(
select *,lead(value,1) over(partition by user_id order by activity_date) as next_col from queries.default.table1
)
, t2 as(
select *,
case when next_col>value or next_col is null then 1 else 0 end as flag
from t1)
select distinct(user_id)
from t2 where user_id not in(
select user_id from t2  
where flag = 0)

user_id
1
4
